# 🧠 BETIX AI Engine Playground (Generic)

Ce notebook teste la version **agnostique** et **réutilisable** de la classe `ChatModel`.\nOn injecte la configuration spécifique (Clés API, Modèle) depuis l'extérieur, sans dépendre de `config.py` dans la classe elle-même.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Ajouter le répertoire backend au path
sys.path.append(os.path.abspath(".."))

load_dotenv("../.env")

# Récupération des clés depuis l'environnement (simulation d'une injection externe)
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
print(f"✅ Clé Gemini chargée : {GEMINI_KEY[:5]}...")

## 1. Initialisation Générique
On instancie `ChatModel` sans qu'il ne connaisse le projet BETIX.

In [ ]:
from app.engine.ai_model import ChatModel

# Instance pour Gemini
ai_gemini = ChatModel(
    provider="gemini",
    api_key=GEMINI_KEY,
    temperature=0.7,
    model_name="gemini-1.5-pro"
)

print(f"🚀 Modèle prêt : {ai_gemini.provider}")

## 2. Génération avec System Prompt Dynamique
Le prompt système est passé à la volée lors de la génération, pas à l'init.

In [ ]:
import asyncio

SYSTEM_PROMPT = "Tu es un assistant sarcastique qui déteste le sport."

async def test_generation():
    # Question simple, mais avec un système prompt fort
    response = await ai_gemini.generate_response(
        message="Qui a gagné la coupe du monde 2018 ?",
        system_instruction=SYSTEM_PROMPT
    )
    print(f"🤖 Réponse : {response}")

await test_generation()

## 3. Gestion de l'Historique Externe
On manipule l'historique manuellement pour voir si la prochaine réponse en tient compte.

In [ ]:
# On force l'ajout d'un message dans l'historique
ai_gemini.add_message("user", "J'adore le tennis !")
ai_gemini.add_message("model", "Pff, courir après une balle jaune, quel ennui.")

async def test_memory():
    # La question suivante devrait déclencher une réponse cohérente avec le dédain précédent
    response = await ai_gemini.generate_response(
        message="Et le football ?",
        system_instruction=SYSTEM_PROMPT # On garde le même persona
    )
    print(f"🤖 Réponse : {response}")

await test_memory()

## 4. Vérification de l'Historique Unifié

In [ ]:
import json
print(json.dumps(ai_gemini.get_history(), indent=2))